# HopSkipJump Attack: Tree vs SVM on Iris
## Does adversarial density detect coverage-gap bias across model types?

Same black-box attack (HopSkipJump) on both Decision Tree and SVM(RBF).
Injects coverage gap bias → trains model → generates adversarial points → OPTICS core density.

**Estimated runtime: ~12 min** for the default grid below.

In [ ]:
import numpy as np, polars as pl, time, warnings
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import OPTICS
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump
from scipy import stats
from itertools import product
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
print('OK')

## 1. Bias Injection (contiguous slab deletion)

In [ ]:
def inject_bias(X, y, target_class, feature_idx, bias):
    if bias == 0:
        return X.copy(), y.copy()
    mask = y == target_class
    order = np.argsort(X[mask][:, feature_idx])
    Xs, ys = X[mask][order], y[mask][order]
    n_keep = max(int(len(Xs) * (1 - bias)), 3)
    return (np.vstack([X[~mask], Xs[-n_keep:]]),
            np.hstack([y[~mask], ys[-n_keep:]]))

## 2. Core Functions

In [ ]:
def opt_density(points, min_samples=3):
    """OPTICS core density = 1/mean(core_distance) per cluster. Returns (density, n_clusters)."""
    if len(points) < min_samples + 1:
        return np.nan, np.nan
    o = OPTICS(min_samples=min_samples, xi=0.05, min_cluster_size=min_samples).fit(points)
    densities = []
    for c in set(o.labels_) - {-1}:
        cd = o.core_distances_[o.labels_ == c]
        if len(cd) > 0 and np.mean(cd) > 0:
            densities.append(1.0 / np.mean(cd))
    return (np.mean(densities), len(densities)) if densities else (np.nan, 0)


def gen_adv(art_model, X, y):
    """HopSkipJump: generate adversarial points for correctly-classified samples.
    Returns (adv_points, success_rate)."""
    preds = np.argmax(art_model.predict(X), axis=1)
    correct = preds == y
    Xc, yc = X[correct], y[correct]
    if len(Xc) == 0:
        return np.array([]), 0.0
    hs = HopSkipJump(classifier=art_model, norm=2, max_iter=10, max_eval=200, init_eval=50, verbose=False)
    adv = hs.generate(Xc)
    ap = np.argmax(art_model.predict(adv), axis=1)
    success = ap != yc
    return adv[success], success.mean()

## 3. Single Experiment

In [ ]:
def run_one(X, y, model_sklearn, target_class, feature_idx, bias, seed, n_folds=5):
    np.random.seed(seed)
    Xb, yb = inject_bias(X, y, target_class, feature_idx, bias)
    ohe = OneHotEncoder(sparse_output=False)
    yb_oh = ohe.fit_transform(yb.reshape(-1, 1))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = []
    for tr, te in skf.split(Xb, yb):
        Xt, Xv, yt, yv = Xb[tr], Xb[te], yb[tr], yb[te]
        m = model_sklearn.__class__(**model_sklearn.get_params()).fit(Xt, yt)
        art = SklearnClassifier(m)
        art.fit(Xt, yb_oh[tr])
        adv, succ = gen_adv(art, Xv, yv)
        d, nc = opt_density(adv)
        folds.append({'tacc': m.score(Xt, yt), 'vacc': m.score(Xv, yv),
                       'asucc': succ, 'nadv': len(adv), 'density': d, 'nclust': nc})
    r = {k: np.nanmean([f[k] for f in folds]) for k in folds[0]}
    r.update(seed=seed, bias=bias, tc=target_class, feat=feature_idx)
    return r


## 4. Run Grid

Adjust `BIAS_LEVELS`, `SEEDS`, `FEATURES` to control runtime.

In [ ]:
# --- CONFIG ---
BIAS_LEVELS = [0.1, 0.3, 0.5, 0.7, 0.9]
SEEDS       = [42, 58, 125]
FEATURES    = [0, 1, 2, 3]  # all iris features
# ---

iris = load_iris()
models = {
    'tree': DecisionTreeClassifier(max_depth=3, random_state=42),
    'svm':  SVC(kernel='rbf', probability=True, random_state=42),
}

CLASSES = [0, 1, 2]
TOTAL = len(BIAS_LEVELS) * len(SEEDS) * len(CLASSES) * len(FEATURES) * len(models)
print(f'Grid: {len(BIAS_LEVELS)}b x {len(SEEDS)}s x {len(CLASSES)}c x {len(FEATURES)}f x {len(models)}m = {TOTAL} runs')

rows = []
t_start = time.time()
for i, (bias, seed, tc, feat, (nm, m)) in enumerate(product(BIAS_LEVELS, SEEDS, CLASSES, FEATURES, models.items())):
    r = run_one(iris.data, iris.target, m, tc, feat, bias, seed, 5)
    r['model'] = nm
    rows.append(r)
    if (i+1) % 60 == 0 or i+1 == TOTAL:
        elapsed = time.time() - t_start
        pct = (i+1) / TOTAL
        eta = elapsed / pct - elapsed
        print(f'[{i+1}/{TOTAL}] {pct:.0%}  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m')

df = pl.DataFrame(rows)
print(f'\nTotal: {time.time()-t_start:.0f}s. Shape: {df.shape}')
df.head()

## 5. Results Table

In [ ]:
agg = df.group_by(['model', 'bias']).agg(
    pl.col('density').mean().alias('density'), pl.col('density').std().alias('d_std'),
    pl.col('nclust').mean().alias('clusters'), pl.col('vacc').mean().alias('test_acc'),
    pl.col('asucc').mean().alias('adv_rate'), pl.col('nadv').mean().alias('nadv'),
).sort(['model', 'bias'])
agg

## 6. Plot: Core Density + Test Accuracy

In [ ]:
df_z = df.with_columns(
    ((pl.col('density') - pl.col('density').mean()) / pl.col('density').std()).over('model').alias('dz')
)
az = df_z.group_by(['model', 'bias']).agg(
    pl.col('dz').mean().alias('z'), pl.col('dz').std().alias('zs'),
    pl.col('dz').count().alias('n'), pl.col('vacc').mean().alias('test_acc'),
).sort(['model', 'bias'])

az = az.with_columns(
    pl.struct(['z','zs','n']).map_elements(
        lambda r: r['z'] - stats.t.ppf(0.975, r['n']-1)*r['zs']/np.sqrt(r['n']),
        return_dtype=pl.Float64).alias('lo'),
    pl.struct(['z','zs','n']).map_elements(
        lambda r: r['z'] + stats.t.ppf(0.975, r['n']-1)*r['zs']/np.sqrt(r['n']),
        return_dtype=pl.Float64).alias('hi'),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, nm in zip(axes, ['tree', 'svm']):
    d = az.filter(pl.col('model') == nm).sort('bias')
    x = d['bias'].to_numpy()
    ax.plot(x, d['test_acc'], 's-', color='#1f77b4', lw=2, ms=6, label='Test Acc')
    ax.set_ylabel('Accuracy', color='#1f77b4'); ax.tick_params(axis='y', labelcolor='#1f77b4')
    ax.set_ylim(0.6, 1.05)
    ax2 = ax.twinx()
    c = '#2ca02c' if nm == 'tree' else '#d62728'
    ax2.fill_between(x, d['lo'], d['hi'], color=c, alpha=0.12)
    ax2.plot(x, d['z'], 'o--', color=c, lw=2, ms=6, label='Density (z)')
    ax2.axhline(0, color='gray', ls=':', lw=1)
    ax2.set_ylabel('Core Density (z)', color=c); ax2.tick_params(axis='y', labelcolor=c)
    ax.set_xlabel('Bias'); ax.set_title(nm.upper(), fontweight='bold'); ax.set_xlim(0.05, 0.95)
    h1,l1 = ax.get_legend_handles_labels(); h2,l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc='lower left', fontsize=9)
fig.suptitle('HopSkipJump: Core Density vs Coverage Gap (iris)', fontsize=14, fontweight='bold')
fig.tight_layout(); plt.show()

## 7. Per-Class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, nm in zip(axes, ['tree', 'svm']):
    for tc in [0,1,2]:
        sub = df_z.filter((pl.col('model')==nm)&(pl.col('tc')==tc))
        c = sub.group_by('bias').agg(pl.col('dz').mean().alias('z')).sort('bias')
        ax.plot(c['bias'], c['z'], 'o-', lw=2, ms=5, label=f'Class {tc}')
    ax.axhline(0, color='gray', ls=':'); ax.set_xlabel('Bias'); ax.set_ylabel('Density (z)')
    ax.set_title(nm.upper(), fontweight='bold'); ax.set_xlim(0.05,0.95); ax.legend()
fig.suptitle('Per-Class Density', fontsize=14, fontweight='bold')
fig.tight_layout(); plt.show()

## 8. Statistical Test

In [ ]:
for nm in ['tree', 'svm']:
    lo = df.filter((pl.col('model')==nm)&(pl.col('bias')==0.1)).group_by(['seed','tc','feat']).agg(pl.col('density').mean())
    hi = df.filter((pl.col('model')==nm)&(pl.col('bias')==0.9)).group_by(['seed','tc','feat']).agg(pl.col('density').mean())
    vl, vh = lo['density'].to_numpy(), hi['density'].to_numpy()
    t, p = stats.ttest_ind(vl, vh)
    d = (np.nanmean(vh) - np.nanmean(vl)) / np.sqrt((np.nanvar(vl) + np.nanvar(vh)) / 2)
    print(f'{nm:>6s}: {np.nanmean(vl):.4f} -> {np.nanmean(vh):.4f}  t={t:.2f}  p={p:.4f}  d={d:.3f}')

---
**Attack:** HopSkipJump (ART 1.20.1), `max_iter=10, max_eval=200, init_eval=50`\
**Density:** OPTICS core distance (`1/mean(core_distance)`)\
**SVM:** `probability=True` (deprecated in sklearn 1.9 but avoids triple-fit overhead of CalibratedClassifierCV)\n**To scale up:** increase BIAS_LEVELS/SEEDS/FEATURES in Cell 4.